# OeNB Web Crawler - Demo

Dieses Notebook zeigt, was der OeNB-Crawler kann und welche Daten er gesammelt hat.

1. **Architektur** - Wie der Crawler funktioniert
2. **Datenbank** - Was wurde gecrawlt (Statistiken)
3. **Inhalte** - Sektionen, Sprachen, Beispielseiten
4. **Zeitreihen-Daten** - Automatisch extrahierte Chart-Daten (isawebstat)
5. **Live-Demo: Agentic Search** - Fragen an die OeNB-Daten stellen

---

## 1. Architektur

Der Crawler basiert auf **Scrapy** und speichert alle Seiten in einer **SQLite-Datenbank**.

```
oenb.at / finanzbildung.oenb.at
        |
        v
  Scrapy Spider          (polite crawling: 0.5s delay, AutoThrottle)
        |
        v
  DeduplicationPipeline  (URL-Normalisierung, Session-IDs entfernen)
        |
        v
  SQLitePipeline         (Speicherung mit gzip-Kompression)
        |
        v
  pages.db               (Seiten + Bodies + extrahierter Text)
        |
        v
  Text-Extraktion        (BeautifulSoup + Chart-Daten aus JavaScript)
        |
        v
  Agentic Search         (SQL-Stichwortsuche + Ollama LLM)
```

**Crawl-Konfiguration:**
- Start-URLs: `oenb.at`, `oenb.at/Service/Sitemap.html`, `finanzbildung.oenb.at`
- Polite Crawling: `robots.txt` wird respektiert, 0.5s Delay, max 4 parallele Requests
- AutoThrottle passt Geschwindigkeit automatisch an Server-Last an
- Inkrementell: Bei erneutem Crawl werden nur geanderte Seiten aktualisiert (SHA256-Hash)

**Datenbank-Schema:**
- `pages` - URL, Status, Content-Type, Zeitstempel, Body-Hash
- `page_bodies` - Komprimierter HTML-Body (gzip)
- `page_content` - Extrahierter Text, Titel, Sektion, Sprache

## 2. Datenbank-Statistiken

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DB_PATH = Path("../data/pages.db")
conn = sqlite3.connect(DB_PATH)

# Basis-Zahlen
pages = conn.execute("SELECT COUNT(*) FROM pages").fetchone()[0]
bodies = conn.execute("SELECT COUNT(*) FROM page_bodies").fetchone()[0]
content = conn.execute("SELECT COUNT(*) FROM page_content").fetchone()[0]

# DB-Dateigrösse
db_size_mb = DB_PATH.stat().st_size / 1024 / 1024

# Crawl-Run Info
runs = pd.read_sql('''
    SELECT id, seed_url, started_at, finished_at, user_agent
    FROM crawl_runs ORDER BY id DESC LIMIT 5
''', conn)

print(f"=== OeNB Crawler - Datenbank ===")
print(f"Datei:       {DB_PATH.resolve()}")
print(f"Groesse:     {db_size_mb:.1f} MB")
print(f"")
print(f"Seiten:      {pages:>7,}")
print(f"Bodies:      {bodies:>7,}")
print(f"Texte:       {content:>7,}")
print(f"")
print(f"=== Letzte Crawl-Runs ===")
display(runs)

=== OeNB Crawler - Datenbank ===
Datei:       /home/christoph/Dokumente/Baumhaus/Programmieren/oenb_downloads_skilled_version/data/pages.db
Groesse:     639.4 MB

Seiten:       10,010
Bodies:       10,010
Texte:        10,010

=== Letzte Crawl-Runs ===


,id,seed_url,started_at,finished_at,user_agent
0,2,https://www.oenb.at/,2026-01-23T13:34:29.705032Z,2026-01-24T01:15:50.947782Z,OeNB-IWG-Audit-Bot/1.0 (Open Data compliance c...
1,1,https://www.oenb.at/Statistik.html,2026-01-23T08:51:38.371498Z,2026-01-23T08:56:12.081491Z,OeNB-IWG-Audit-Bot/1.0 (Open Data compliance c...


In [2]:
# Text-Statistiken
stats = pd.read_sql('''
    SELECT
        COUNT(*) as seiten,
        ROUND(AVG(LENGTH(text_content)), 0) as avg_laenge,
        MIN(LENGTH(text_content)) as min_laenge,
        MAX(LENGTH(text_content)) as max_laenge,
        SUM(LENGTH(text_content)) as total_zeichen
    FROM page_content
    WHERE LENGTH(text_content) > 0
''', conn)

total_chars = stats['total_zeichen'].iloc[0]
total_words = total_chars / 5  # grobe Schätzung

print(f"=== Text-Extraktion ===")
print(f"Seiten mit Text:    {stats['seiten'].iloc[0]:>8,}")
print(f"Gesamt-Zeichen:     {total_chars:>8,.0f}  (~{total_words:,.0f} Wörter)")
print(f"Ø Textlaenge:       {stats['avg_laenge'].iloc[0]:>8,.0f} Zeichen")
print(f"Max Textlaenge:     {stats['max_laenge'].iloc[0]:>8,} Zeichen")

# Qualität
quality = pd.read_sql('''
    SELECT
        CASE
            WHEN LENGTH(text_content) = 0 THEN '1. Leer'
            WHEN LENGTH(text_content) < 100 THEN '2. Fast leer (<100)'
            WHEN LENGTH(text_content) < 500 THEN '3. Kurz (<500)'
            WHEN LENGTH(text_content) > 100000 THEN '5. Sehr lang (>100K)'
            ELSE '4. Normal (500-100K)'
        END as kategorie,
        COUNT(*) as anzahl
    FROM page_content
    GROUP BY kategorie
    ORDER BY kategorie
''', conn)

print(f"\n=== Seitenqualitaet ===")
total = quality['anzahl'].sum()
for _, row in quality.iterrows():
    pct = row['anzahl'] / total * 100
    bar = '#' * int(pct / 2)
    print(f"  {row['kategorie']:30s} {row['anzahl']:>6,}  ({pct:4.1f}%)  {bar}")

=== Text-Extraktion ===
Seiten mit Text:       9,982
Gesamt-Zeichen:     55,485,013  (~11,097,003 Wörter)
Ø Textlaenge:          5,559 Zeichen
Max Textlaenge:      788,095 Zeichen

=== Seitenqualitaet ===
  1. Leer                            28  ( 0.3%)  
  2. Fast leer (<100)                98  ( 1.0%)  
  3. Kurz (<500)                  1,807  (18.1%)  #########
  4. Normal (500-100K)            7,980  (79.7%)  #######################################
  5. Sehr lang (>100K)               97  ( 1.0%)  


## 3. Inhalte

In [3]:
# Sektionen
sections = pd.read_sql('''
    SELECT page_section as sektion, COUNT(*) as seiten,
           ROUND(AVG(LENGTH(text_content)), 0) as avg_text
    FROM page_content
    GROUP BY page_section
    ORDER BY seiten DESC
''', conn)

print(f"=== {len(sections)} Sektionen ===")
print()
for _, row in sections.head(15).iterrows():
    bar = '#' * (row['seiten'] // 50)
    print(f"  {row['sektion']:30s} {row['seiten']:>5,} Seiten  (Ø {row['avg_text']:,.0f} Zeichen)  {bar}")

=== 111 Sektionen ===

  isawebstat                     2,182 Seiten  (Ø 2,764 Zeichen)  ###########################################
  Presse                         1,152 Seiten  (Ø 3,842 Zeichen)  #######################
  Publikationen                  1,125 Seiten  (Ø 15,373 Zeichen)  ######################
  Publications                     841 Seiten  (Ø 17,262 Zeichen)  ################
  meldewesen                       591 Seiten  (Ø 1,720 Zeichen)  ###########
  Ueber-Uns                        482 Seiten  (Ø 2,263 Zeichen)  #########
  Termine                          473 Seiten  (Ø 1,543 Zeichen)  #########
  Statistik                        441 Seiten  (Ø 2,537 Zeichen)  ########
  Statistics                       420 Seiten  (Ø 1,435 Zeichen)  ########
  Calendar                         257 Seiten  (Ø 1,784 Zeichen)  #####
  Kontakt                          209 Seiten  (Ø 516 Zeichen)  ####
  Media                            209 Seiten  (Ø 3,406 Zeichen)  ####
  About-Us 

In [4]:
# Sprachen
langs = pd.read_sql('''
    SELECT language as sprache, COUNT(*) as seiten
    FROM page_content GROUP BY language ORDER BY seiten DESC
''', conn)

print("=== Sprachen ===")
for _, row in langs.iterrows():
    pct = row['seiten'] / pages * 100
    print(f"  {row['sprache']}: {row['seiten']:>6,}  ({pct:.0f}%)")

=== Sprachen ===
  de:  7,556  (75%)
  en:  2,454  (25%)


In [5]:
# 5 Beispielseiten aus verschiedenen Sektionen
examples = pd.read_sql('''
    SELECT p.url, pc.title, pc.page_section,
           LENGTH(pc.text_content) as text_len,
           SUBSTR(pc.text_content, 1, 200) as vorschau
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE LENGTH(pc.text_content) BETWEEN 500 AND 10000
      AND pc.page_section IN ('Geldpolitik', 'Statistik', 'Presse', 'Publikationen', 'FAQ')
    GROUP BY pc.page_section
    ORDER BY RANDOM()
    LIMIT 5
''', conn)

print("=== Beispielseiten ===")
for i, (_, row) in enumerate(examples.iterrows(), 1):
    print(f"\n--- [{row['page_section']}] {row['title'][:60]} ---")
    print(f"URL:  {row['url']}")
    print(f"Text: {row['vorschau']}...")

=== Beispielseiten ===

--- [Presse] Schützt Finanzbildung vor der Schuldenfalle? - Oesterreichis ---
URL:  https://www.oenb.at/Presse/die-nationalbank-der-podcast/oenb-podcast-folge-46.html
Text: Schützt Finanzbildung vor der Schuldenfalle? - Oesterreichische Nationalbank (OeNB) Zur Navigation Zum Inhalt Schützt Finanzbildung vor der Schuldenfalle? Folge 46 des Nationalbank-Podcasts (01.08.202...

--- [Publikationen] OeNB Report 2024/13: International Survey of Adult Financial ---
URL:  https://www.oenb.at/Publikationen/Volkswirtschaft/reports/2024/report-2024-13-financial-literacy-asfl/html-version-de.html
Text: OeNB Report 2024/13: International Survey of Adult Financial Literacy 2023: erste Ergebnisse für Österreich – Executive Summary - Oesterreichische Nationalbank (OeNB) Zur Navigation Zum Inhalt Aktuell...

--- [Geldpolitik] East Jour Fixe - Oesterreichische Nationalbank (OeNB) ---
URL:  https://www.oenb.at/Geldpolitik/schwerpunkt-zentral-ost-und-suedosteuropa-cesee/veranstaltu

## 4. Zeitreihen-Daten (isawebstat)

Die OeNB stellt interaktive Charts auf `isawebstat` bereit (Leitzinssaetze, Kreditzinssaetze, Versicherungsstatistik, etc.).

Diese Charts laden ihre Daten per JavaScript (`$scope.data = [...]`). Der Crawler extrahiert diese Zeitreihen automatisch und macht sie durchsuchbar.

In [ ]:
# Wie viele Chart-Seiten haben wir?
chart_pages = pd.read_sql('''
    SELECT COUNT(*) as total,
           SUM(CASE WHEN pc.text_content LIKE '%, 2024:%' THEN 1 ELSE 0 END) as mit_daten
    FROM page_content pc
    WHERE pc.title LIKE 'DATA Chart%'
''', conn)

total_isawebstat = pd.read_sql('''
    SELECT COUNT(*) as c FROM page_content WHERE page_section = 'isawebstat'
''', conn).iloc[0, 0]

print(f"=== isawebstat Seiten ===")
print(f"Gesamt isawebstat:        {total_isawebstat:>5,}")
print(f"Davon Chart-Seiten:       {chart_pages['total'].iloc[0]:>5,}")
print(f"Mit extrahierten Daten:   {chart_pages['mit_daten'].iloc[0]:>5,}")

In [ ]:
# Welche Charts haben wir?
charts = pd.read_sql('''
    SELECT REPLACE(pc.title, 'DATA Chart - ', '') as chart_name,
           p.url,
           LENGTH(pc.text_content) as text_len
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE pc.title LIKE 'DATA Chart%'
      AND pc.text_content LIKE '%, 2024:%'
    ORDER BY chart_name
    LIMIT 30
''', conn)

print(f"=== Charts mit Zeitreihen-Daten (Auswahl, {len(charts)} gezeigt) ===")
for _, row in charts.iterrows():
    print(f"  {row['chart_name'][:70]}")

In [ ]:
# Beispiel: Leitzinssätze - die extrahierten Zeitreihen
leitzins = pd.read_sql('''
    SELECT pc.title, pc.text_content
    FROM page_content pc
    WHERE pc.title LIKE 'DATA Chart - Leitzins%'
    LIMIT 1
''', conn)

if len(leitzins) > 0:
    text = leitzins.iloc[0]['text_content']
    title = leitzins.iloc[0]['title']
    # Die Zeitreihen stehen am Ende des Texts (nach der Navigation)
    # Finde die erste Zeile mit "Jahr: Wert" Muster
    lines = text.split('\n')
    data_lines = [l for l in lines if ': 2024:' in l or '(Quelle:' in l]
    if data_lines:
        print(f"=== {title} ===")
        print()
        for line in data_lines:
            # Nur die letzten 10 Jahre zeigen
            if ': 2015:' in line:
                pos = line.find('2015:')
                key = line[:pos].strip().rstrip(',').strip()
                print(f"{key} {line[pos:]}")
            elif '(Quelle:' in line:
                print(line)
    else:
        # Fallback: letzte 600 Zeichen
        print(f"=== {title} ===")
        print()
        print(text[-600:])
else:
    print("Keine Leitzins-Chart-Seite gefunden")

## 5. Live-Demo: Agentic Search

**Agentic Search** = SQL-Stichwortsuche + Ollama LLM. Kein Embedding, kein Vector Store, kein LangChain.

**Voraussetzung:** Ollama muss laufen (`ollama serve`) mit `llama3.1:8b`.

In [ ]:
import time
import json
import requests

# Synonym-Dictionary laden fuer bessere Chart-Suche
with open("chart_synonyms.json") as f:
    SYNONYMS = json.load(f)
    del SYNONYMS["_info"]


def expand_keywords(keywords):
    """Keywords um Synonyme erweitern. Wenn ein Keyword als Synonym vorkommt,
    wird der zugehoerige Chart-Begriff als zusaetzliches Keyword hinzugefuegt."""
    expanded = list(keywords)
    for kw in keywords:
        kw_lower = kw.lower()
        # Prüfe ob das Keyword in einem Synonym-Eintrag vorkommt
        for chart_term, syns in SYNONYMS.items():
            if any(kw_lower in s.lower() for s in syns) and chart_term not in expanded:
                expanded.append(chart_term)
            # Oder ob das Keyword ein Chart-Begriff ist -> Synonyme hinzufuegen
            if kw_lower in chart_term.lower():
                for s in syns[:3]:  # Max 3 Synonyme pro Treffer
                    if s not in expanded:
                        expanded.append(s)
    return expanded


def search_sqlite(keywords, limit=5):
    """Stichwortsuche: erstes Keyword required, weitere boosten Ranking."""
    # Keywords um Synonyme erweitern
    all_keywords = expand_keywords(keywords)

    # Erstes Original-Keyword ist Pflicht
    where = f"pc.text_content LIKE '%{keywords[0]}%'"

    # Alle weiteren Keywords (inkl. Synonyme) als Ranking-Boost
    bonus_expressions = []
    for kw in all_keywords[1:]:
        bonus_expressions.append(
            f"(CASE WHEN pc.text_content LIKE '%{kw}%' THEN 0 ELSE 1 END)"
        )
    bonus_order = ", ".join(bonus_expressions) if bonus_expressions else "0"

    query = f'''
        SELECT p.url, pc.title, pc.page_section,
               pc.text_content,
               LENGTH(pc.text_content) as text_len
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE {where}
          AND LENGTH(pc.text_content) > 200
        ORDER BY
          -- Chart-Seiten mit Zeitreihen bevorzugen
          CASE WHEN pc.title LIKE 'DATA Chart%' THEN 0 ELSE 1 END,
          -- Bonus-Keywords matchen (je mehr desto besser)
          {bonus_order},
          -- Keyword im Titel
          CASE WHEN pc.title LIKE '%{keywords[0]}%' THEN 0 ELSE 1 END,
          -- Keyword in URL
          CASE WHEN p.url LIKE '%{keywords[0].lower()}%' THEN 0 ELSE 1 END,
          -- Mittlere Seitenlänge bevorzugen
          CASE
            WHEN LENGTH(pc.text_content) BETWEEN 1000 AND 20000 THEN 0
            WHEN LENGTH(pc.text_content) BETWEEN 500 AND 1000 THEN 1
            ELSE 2
          END,
          text_len ASC
        LIMIT {limit}
    '''
    df_results = pd.read_sql(query, conn)

    # Snippet-Extraktion
    snippets = []
    main_kw = keywords[0]
    for _, row in df_results.iterrows():
        text = row['text_content']
        title = row['title'] or ''

        # Chart-Seiten: die Zeitreihen-Daten stehen am Ende
        if title.startswith('DATA Chart'):
            snippet = text[-800:]
        else:
            pos = text.lower().find(main_kw.lower())
            if pos >= 0:
                start = max(0, pos - 300)
                end = min(len(text), pos + 500)
                snippet = ("..." if start > 0 else "") + text[start:end] + ("..." if end < len(text) else "")
            else:
                snippet = text[:800]
        snippets.append(snippet)
    df_results['snippet'] = snippets

    return df_results


def ask_ollama(prompt, model="llama3.1:8b"):
    """Ollama direkt via HTTP API."""
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
    })
    return resp.json()["response"]


def frage(question: str, keywords: list):
    """Frage an die OeNB-Daten stellen."""
    expanded = expand_keywords(keywords)

    print("=" * 70)
    print(f"FRAGE: {question}")
    print(f"SUCHBEGRIFFE: {keywords}")
    if len(expanded) > len(keywords):
        print(f"ERWEITERT UM: {expanded[len(keywords):]}")
    print("=" * 70)

    t0 = time.time()
    results = search_sqlite(keywords)
    search_time = time.time() - t0

    total = pd.read_sql(
        f"SELECT COUNT(*) as c FROM page_content WHERE text_content LIKE '%{keywords[0]}%'",
        conn
    ).iloc[0, 0]

    print(f"\n{total} Treffer, Top {len(results)} in {search_time:.3f}s")

    if len(results) == 0:
        print("Keine Treffer.")
        return

    context = "\n\n---\n\n".join(
        f"Titel: {row['title']}\nURL: {row['url']}\nText: {row['snippet']}"
        for _, row in results.iterrows()
    )

    prompt = f"""Beantworte die Frage basierend auf dem folgenden Kontext.
Wenn du die Antwort nicht im Kontext findest, sag das ehrlich.
Antworte auf Deutsch.

Kontext:
{context}

Frage: {question}

Antwort:"""

    t0 = time.time()
    answer = ask_ollama(prompt)
    llm_time = time.time() - t0

    print(f"\nANTWORT ({llm_time:.1f}s):\n{answer}")
    print("\n" + "-" * 70)
    print("QUELLEN:")
    for i, (_, row) in enumerate(results.iterrows(), 1):
        print(f"  [{i}] {row['title'][:55]}")
        print(f"      {row['url']}")
    print("=" * 70)


print("Agentic Search bereit (mit Synonym-Erweiterung).")

In [ ]:
frage("Was ist der aktuelle Leitzins der EZB?", ["Leitzins", "EZB"])

FRAGE: Was ist der aktuelle Leitzins der EZB?
SUCHBEGRIFFE: ['Leitzins', 'EZB']

121 Treffer, Top 5 in 0.090s


In [ ]:
frage("Wie haben sich die Kreditzinssaetze entwickelt?", ["Kreditzins"])

In [ ]:
frage("Was macht die OeNB im Bereich Finanzbildung?", ["Finanzbildung"])

In [ ]:
# Eigene Frage:
# frage("Deine Frage hier", ["Suchbegriff1", "Suchbegriff2"])

---

## Zusammenfassung

| | |
|---|---|
| **Crawler** | Scrapy-basiert, polite crawling, inkrementell |
| **Seiten** | ~10.000 (oenb.at + finanzbildung.oenb.at) |
| **Sprachen** | DE (~75%) + EN (~25%) |
| **Sektionen** | 30+ (Statistik, Presse, Publikationen, Geldpolitik, ...) |
| **Zeitreihen** | ~1.200 Charts mit numerischen Daten (Leitzins, Kreditzins, ...) |
| **Suche** | Agentic Search: SQL-Keywords + Ollama LLM |
| **Dependencies** | SQLite + requests + Ollama (kein LangChain/ChromaDB noetig) |